# Customer Spending Score Regression
IF3270 Machine Learning - Practicum 1

Steven Owen - 13523103
Daniel Pedrosa Wu - 13523099

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, MaxAbsScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error,
)
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor,
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.base import clone
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load Data

In [ ]:
TARGET      = "Family_Size"             # Regression target (numeric column)
ID_COL      = "ID"
TRAIN_PATH  = "Train_processed.csv"
TEST_PATH   = "test_processed_no_solution.csv"
SCORING     = "neg_root_mean_squared_error"   # sklearn convention (higher = better)

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

df_train.head()

---
## 3. Exploratory Data Analysis
### 3.1 Data Integrity

In [ ]:
print(f"Train shape: {df_train.shape}")
print(f"Test  shape: {df_test.shape}")
print(f"\nDuplicate rows (train): {df_train.duplicated().sum()}")
print(f"Duplicate rows (test):  {df_test.duplicated().sum()}")

df_train.info()

In [ ]:
missing = df_train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    missing.plot.barh(ax=ax, color="salmon")
    ax.set_title("Missing Values per Feature")
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values.")

### 3.2 Feature Separation (Dynamic)

In [ ]:
feature_cols = [c for c in df_train.columns if c not in [ID_COL, TARGET]]

num_cols = df_train[feature_cols].select_dtypes(include="number").columns.tolist()
cat_cols = df_train[feature_cols].select_dtypes(exclude="number").columns.tolist()

print(f"Numerical  ({len(num_cols)}): {num_cols}")
print(f"Categorical ({len(cat_cols)}): {cat_cols}")

### 3.3 Univariate Analysis

In [ ]:
for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    sns.histplot(df_train[col], kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {col}")
    sns.boxplot(x=df_train[col], ax=axes[1])
    axes[1].set_title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in cat_cols:
    fig, ax = plt.subplots(figsize=(8, 3))
    order = df_train[col].value_counts().index
    sns.countplot(data=df_train, x=col, order=order, ax=ax)
    ax.set_title(f"Count of {col}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(df_train[TARGET], kde=True, ax=ax, color="teal", bins=20)
ax.set_title(f"Target Distribution — {TARGET}")
plt.tight_layout()
plt.show()

df_train[TARGET].describe()

### 3.4 Multivariate Analysis

In [ ]:
corr_cols = num_cols + [TARGET]
if len(corr_cols) > 1:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        df_train[corr_cols].corr(),
        annot=True, fmt=".2f", cmap="coolwarm", ax=ax, vmin=-1, vmax=1
    )
    ax.set_title("Correlation Heatmap (Numerical Features + Target)")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in num_cols:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.scatterplot(data=df_train, x=col, y=TARGET, alpha=0.4, ax=ax)
    ax.set_title(f"{col} vs {TARGET}")
    plt.tight_layout()
    plt.show()

for col in cat_cols:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=df_train, x=col, y=TARGET, ax=ax, palette="Set2")
    ax.set_title(f"{col} vs {TARGET}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

### Question 1

*Your answer here…*

### Question 2

*Your answer here…*

### Question 3

*Your answer here…*

---
## 4. Data Preprocessing

In [ ]:
COLUMNS_TO_DROP = ["Work_Experience"]
df_train.drop(columns=[c for c in COLUMNS_TO_DROP if c in df_train.columns], inplace=True)
df_test.drop(columns=[c for c in COLUMNS_TO_DROP if c in df_test.columns], inplace=True)

num_cols = [c for c in num_cols if c not in COLUMNS_TO_DROP]
cat_cols = [c for c in cat_cols if c not in COLUMNS_TO_DROP]
feature_cols = [c for c in feature_cols if c not in COLUMNS_TO_DROP]

In [ ]:
# def remove_outliers_iqr(data, cols):
#     mask = pd.Series(True, index=data.index)
#     for col in cols:
#         Q1  = data[col].quantile(0.25)
#         Q3  = data[col].quantile(0.75)
#         IQR = Q3 - Q1
#         mask &= data[col].between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
#     return data[mask]

# before = df_train.shape[0]
# df_train = remove_outliers_iqr(df_train, num_cols)
# print(f"Rows removed: {before - df_train.shape[0]}  ({before} → {df_train.shape[0]})")

In [ ]:
SCALER = StandardScaler()

In [ ]:
df = df_train.copy()

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Drop rows where target is NaN
df.dropna(subset=[TARGET], inplace=True)

df.isnull().sum().sum()

In [ ]:
ordinal_cols = [c for c in cat_cols if df[c].nunique() <= 5]
onehot_cols  = [c for c in cat_cols if df[c].nunique() > 5]

preprocessor = ColumnTransformer(
    transformers=[
        # ("num", SCALER, num_cols),
        ("num", "passthrough", num_cols),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ordinal_cols),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), onehot_cols),
    ],
    remainder="drop",
)

print(f"Ordinal-encoded: {ordinal_cols}")
print(f"One-Hot-encoded:  {onehot_cols}")

In [ ]:
X = df[feature_cols]
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42,
)

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}")
print(f"Target stats (train): mean={y_train.mean():.4f}, std={y_train.std():.4f}")

### Preprocessing Rationale

*Explain your preprocessing decisions here…*

---
## 5. Modeling — Ensemble Learning (Regression)

We evaluate six ensemble strategies: **Bagging** (Random Forest), **Boosting** (Gradient Boosting, XGBoost, LightGBM), **Voting**, and **Stacking**.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

def eval_regression(y_true, y_pred, label=""):
    """Print common regression metrics."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"[{label}]  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  MAPE={mape:.4f}")
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

### 5.1 Bagging — Random Forest

In [ ]:
pipe_rf = Pipeline([
    ("pre", preprocessor),
    ("reg", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])

scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring=SCORING)
results["Random Forest"] = -scores_rf.mean()   # negate back to positive RMSE

pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_val)

print(f"CV {SCORING}: {scores_rf.mean():.4f} ± {scores_rf.std():.4f}")
eval_regression(y_val, y_pred_rf, "RF Val")

### 5.2 Boosting — Gradient Boosting

In [ ]:
pipe_gb = Pipeline([
    ("pre", preprocessor),
    ("reg", GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)),
])

scores_gb = cross_val_score(pipe_gb, X_train, y_train, cv=cv, scoring=SCORING)
results["Gradient Boosting"] = -scores_gb.mean()

pipe_gb.fit(X_train, y_train)
y_pred_gb = pipe_gb.predict(X_val)

print(f"CV {SCORING}: {scores_gb.mean():.4f} ± {scores_gb.std():.4f}")
eval_regression(y_val, y_pred_gb, "GB Val")

### 5.3 Boosting — XGBoost

In [ ]:
pipe_xgb = Pipeline([
    ("pre", preprocessor),
    ("reg", XGBRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        random_state=42, n_jobs=-1, verbosity=0,
    )),
])

scores_xgb = cross_val_score(pipe_xgb, X_train, y_train, cv=cv, scoring=SCORING)
results["XGBoost"] = -scores_xgb.mean()

pipe_xgb.fit(X_train, y_train)
y_pred_xgb = pipe_xgb.predict(X_val)

print(f"CV {SCORING}: {scores_xgb.mean():.4f} ± {scores_xgb.std():.4f}")
eval_regression(y_val, y_pred_xgb, "XGB Val")

### 5.4 Boosting — LightGBM

In [ ]:
pipe_lgbm = Pipeline([
    ("pre", preprocessor),
    ("reg", LGBMRegressor(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        random_state=42, n_jobs=-1, verbose=-1,
    )),
])

scores_lgbm = cross_val_score(pipe_lgbm, X_train, y_train, cv=cv, scoring=SCORING)
results["LightGBM"] = -scores_lgbm.mean()

pipe_lgbm.fit(X_train, y_train)
y_pred_lgbm = pipe_lgbm.predict(X_val)

print(f"CV {SCORING}: {scores_lgbm.mean():.4f} ± {scores_lgbm.std():.4f}")
eval_regression(y_val, y_pred_lgbm, "LGBM Val")

### 5.5 Voting — Heterogeneous Ensemble

In [ ]:
estimators_vote = [
    ("rf",  Pipeline([("pre", preprocessor), ("reg", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))])),
    ("gb",  Pipeline([("pre", preprocessor), ("reg", GradientBoostingRegressor(n_estimators=200, random_state=42))])),
    ("dt",  Pipeline([("pre", preprocessor), ("reg", DecisionTreeRegressor(max_depth=8, random_state=42))])),
]

pipe_vote = VotingRegressor(
    estimators=estimators_vote,
    weights=[2, 2, 1],
)

scores_vote = cross_val_score(pipe_vote, X_train, y_train, cv=cv, scoring=SCORING)
results["Voting"] = -scores_vote.mean()

pipe_vote.fit(X_train, y_train)
y_pred_vote = pipe_vote.predict(X_val)

print(f"CV {SCORING}: {scores_vote.mean():.4f} ± {scores_vote.std():.4f}")
eval_regression(y_val, y_pred_vote, "Voting Val")

### 5.6 Stacking (passthrough) — Meta-Learner

In [ ]:
estimators_stack = [
    ("rf", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ("gb", GradientBoostingRegressor(n_estimators=200, random_state=42)),
    ("dt", DecisionTreeRegressor(max_depth=8, random_state=42)),
]

pipe_stack_pt = Pipeline([
    ("pre", preprocessor),
    ("reg", StackingRegressor(
        estimators=estimators_stack,
        final_estimator=make_pipeline(
            MaxAbsScaler(),
            Ridge(alpha=1.0),
        ),
        cv=cv,
        n_jobs=-1,
        passthrough=True,
    )),
])

scores_stack_pt = cross_val_score(pipe_stack_pt, X_train, y_train, cv=cv, scoring=SCORING)
results["Stacking (PT)"] = -scores_stack_pt.mean()

pipe_stack_pt.fit(X_train, y_train)
y_pred_stack_pt = pipe_stack_pt.predict(X_val)

print(f"CV {SCORING}: {scores_stack_pt.mean():.4f} ± {scores_stack_pt.std():.4f}")
eval_regression(y_val, y_pred_stack_pt, "Stack-PT Val")

### 5.7 Stacking (no passthrough) — Meta-Learner

In [ ]:
estimators_stack = [
    ("rf", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ("gb", GradientBoostingRegressor(n_estimators=200, random_state=42)),
    ("dt", DecisionTreeRegressor(max_depth=8, random_state=42)),
]

pipe_stack = Pipeline([
    ("pre", preprocessor),
    ("reg", StackingRegressor(
        estimators=estimators_stack,
        final_estimator=Ridge(alpha=1.0),
        cv=cv,
        n_jobs=-1,
        passthrough=False,
    )),
])

scores_stack = cross_val_score(pipe_stack, X_train, y_train, cv=cv, scoring=SCORING)
results["Stacking"] = -scores_stack.mean()

pipe_stack.fit(X_train, y_train)
y_pred_stack = pipe_stack.predict(X_val)

print(f"CV {SCORING}: {scores_stack.mean():.4f} ± {scores_stack.std():.4f}")
eval_regression(y_val, y_pred_stack, "Stack Val")

---
## 6. Evaluation — Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": results.keys(),
    "CV RMSE": [round(v, 4) for v in results.values()],
}).sort_values("CV RMSE", ascending=True).reset_index(drop=True)

comparison.style.highlight_min(subset=["CV RMSE"], color="lightgreen")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=comparison, x="CV RMSE", y="Model", palette="viridis", ax=ax)
ax.set_title("Model Comparison — CV RMSE (lower is better)")
plt.tight_layout()
plt.show()

---
## 7. Error Analysis

In [ ]:
best_name = comparison.iloc[0]["Model"]
best_models = {
    "Random Forest": (pipe_rf, y_pred_rf),
    "Gradient Boosting": (pipe_gb, y_pred_gb),
    "XGBoost": (pipe_xgb, y_pred_xgb),
    "LightGBM": (pipe_lgbm, y_pred_lgbm),
    "Voting": (pipe_vote, y_pred_vote),
    "Stacking (PT)": (pipe_stack_pt, y_pred_stack_pt),
    "Stacking": (pipe_stack, y_pred_stack),
}
best_pipe, y_pred_best = best_models[best_name]

# --- Residual Plot ---
residuals = y_val - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs Actual
axes[0].scatter(y_val, y_pred_best, alpha=0.4, edgecolors="k", linewidths=0.3)
lims = [min(y_val.min(), y_pred_best.min()), max(y_val.max(), y_pred_best.max())]
axes[0].plot(lims, lims, "r--", linewidth=1)
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].set_title(f"Predicted vs Actual: {best_name}")

# Residual Distribution
sns.histplot(residuals, kde=True, ax=axes[1], color="salmon")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual (Actual − Predicted)")

# Residuals vs Predicted
axes[2].scatter(y_pred_best, residuals, alpha=0.4, edgecolors="k", linewidths=0.3)
axes[2].axhline(0, color="r", linestyle="--", linewidth=1)
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Residual")
axes[2].set_title("Residuals vs Predicted")

plt.tight_layout()
plt.show()

print(f"\n--- Best Model: {best_name} ---")
eval_regression(y_val, y_pred_best, best_name)

In [ ]:
# Worst predictions (highest absolute error)
errors = pd.DataFrame({
    "Actual": y_val.values,
    "Predicted": y_pred_best,
    "AbsError": np.abs(residuals.values),
}, index=X_val.index)

worst = errors.nlargest(10, "AbsError")
worst_samples = X_val.loc[worst.index].copy()
worst_samples["Actual"]    = worst["Actual"].values
worst_samples["Predicted"] = worst["Predicted"].round(4).values
worst_samples["AbsError"]  = worst["AbsError"].round(4).values

print(f"Top-10 worst predictions (highest absolute error):")
worst_samples

### 7.1 Hyperparameter Tuning - Best Model

In [ ]:
# param_grids = {
#     "Random Forest": {
#         "reg__n_estimators":      randint(100, 600),
#         "reg__max_depth":         [None, 5, 10, 20, 30],
#         "reg__min_samples_split": randint(2, 20),
#         "reg__min_samples_leaf":  randint(1, 10),
#         "reg__max_features":      ["sqrt", "log2", None],
#     },
#     "Gradient Boosting": {
#         "reg__n_estimators":      randint(100, 500),
#         "reg__learning_rate":     uniform(0.01, 0.3),
#         "reg__max_depth":         randint(3, 10),
#         "reg__subsample":         uniform(0.5, 0.5),
#         "reg__min_samples_split": randint(2, 20),
#     },
#     "XGBoost": {
#         "reg__n_estimators":     randint(100, 500),
#         "reg__learning_rate":    uniform(0.01, 0.3),
#         "reg__max_depth":        randint(3, 10),
#         "reg__subsample":        uniform(0.5, 0.5),
#         "reg__colsample_bytree": uniform(0.5, 0.5),
#         "reg__gamma":            uniform(0, 0.5),
#     },
#     "LightGBM": {
#         "reg__n_estimators":     randint(100, 500),
#         "reg__learning_rate":    uniform(0.01, 0.3),
#         "reg__max_depth":        randint(3, 10),
#         "reg__num_leaves":       randint(20, 100),
#         "reg__subsample":        uniform(0.5, 0.5),
#         "reg__colsample_bytree": uniform(0.5, 0.5),
#     },
#     "Voting": {
#         "weights": [
#             [1, 1, 1], [2, 1, 1], [1, 2, 1], [1, 1, 2],
#             [2, 2, 1], [3, 2, 1], [2, 3, 1], [3, 3, 1],
#         ],
#     },
#     "Stacking (PT)": {
#         "reg__final_estimator__ridge__alpha": uniform(0.01, 10),
#     },
#     "Stacking": {
#         "reg__final_estimator__alpha": uniform(0.01, 10),
#     },
# }

# N_ITER = 10
# tuner = RandomizedSearchCV(
#     clone(best_pipe),
#     param_distributions=param_grids[best_name],
#     n_iter=N_ITER,
#     scoring=SCORING,
#     cv=cv,
#     random_state=42,
#     n_jobs=-1,
#     verbose=3,
# )
# tuner.fit(X_train, y_train)

# print(f"Best CV {SCORING}: {tuner.best_score_:.4f}")
# print(f"Best params:\n{tuner.best_params_}")

# best_pipe = tuner.best_estimator_
# y_pred_best = best_pipe.predict(X_val)

In [ ]:
# eval_regression(y_val, y_pred_best, f"{best_name} (tuned)")

### 7.2 Feature Importance Analysis

In [ ]:
feat_names = preprocessor.get_feature_names_out().tolist()

importances = None

if hasattr(best_pipe, "named_steps") and hasattr(best_pipe.named_steps.get("reg", None), "feature_importances_"):
    importances = best_pipe.named_steps["reg"].feature_importances_
else:
    importances = pipe_rf.named_steps["reg"].feature_importances_

fi = pd.DataFrame({"Feature": feat_names, "Importance": importances})
fi = fi.sort_values("Importance", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=fi, x="Importance", y="Feature", palette="viridis", ax=ax)
ax.set_title(f"Top-20 Feature Importance — {best_name}")
plt.tight_layout()
plt.show()

---
## 8. Kaggle Submission

In [ ]:
pipe_final = clone(best_pipe)

X_full = df[feature_cols]
y_full = df[TARGET]
pipe_final.fit(X_full, y_full)

df_test_clean = df_test.copy()
for col in num_cols:
    if col in df_test_clean.columns:
        df_test_clean[col].fillna(df[col].median(), inplace=True)
for col in cat_cols:
    if col in df_test_clean.columns:
        df_test_clean[col].fillna(df[col].mode()[0], inplace=True)

test_preds = pipe_final.predict(df_test_clean[feature_cols])

submission = pd.DataFrame({
    ID_COL: df_test[ID_COL],
    TARGET: test_preds,
})

submission.to_csv("submission_regression.csv", index=False)
print(f"Submission saved — {submission.shape[0]} rows")
submission.head()